# ORBIT — Phase 3A: Real-Game Gameplay Collection & Verification

**Objective (Step 1 of 5):** Execute real gameplay rollouts in unmodified Crafter using the Gate 6 H=4 policy candidate with controlled exploration ($\tau=1.2$).  
- Record 30 real episodes ($\le 300$ steps/ep) using the pure visual interface `act(obs_uint8, reward_prev, step)`.
- Validate transition schema boundaries and ensure zero simulator privilege leaks.
- Save replay shard to local disk and automatically sync to Google Drive.
- Reload from disk and verify exact trajectory reconstruction.
- **Zero model weights are updated during this stage.**

---

### Cell 1: Environment & Hardware Diagnostics

In [ ]:
# Cell 1: Hardware Diagnostics
import os
import sys
import platform
import torch

print("=" * 75)
print("  SYSTEM & HARDWARE DIAGNOSTICS")
print("=" * 75)
print(f"Python Version:   {platform.python_version()} ({sys.executable})")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:       {vram_gb:.2f} GB")
else:
    print("Running on CPU mode. (Tip: For fast rollouts, select Runtime -> Change runtime type -> T4 GPU)")
print("=" * 75)

### Cell 2: Google Drive Mounting & Namespace Setup
Auto-detects Google Drive (`MyDrive` vs `My Drive`) and initializes the dedicated `learn-a-world/orbit/` persistent storage tree.

In [ ]:
# Cell 2: Robust Drive Mount & Namespace Isolation
import os
from pathlib import Path

# 1. Mount Google Drive safely
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive') and not os.path.exists('/content/drive/My Drive'):
        print("[Drive] Mounting Google Drive to /content/drive...")
        drive.mount('/content/drive')
    else:
        print("[Drive] Google Drive is already mounted.")
except ImportError:
    print("[Environment] Local or non-Colab environment detected.")

# 2. Detect Drive Root Path dynamically
if os.path.exists('/content/drive/MyDrive'):
    DRIVE_ROOT = Path('/content/drive/MyDrive/learn-a-world')
elif os.path.exists('/content/drive/My Drive'):
    DRIVE_ROOT = Path('/content/drive/My Drive/learn-a-world')
else:
    DRIVE_ROOT = Path('/content/learn-a-world/drive_backup')

# 3. Dedicated ORBIT Subdirectories
ORBIT_DRIVE = DRIVE_ROOT / 'orbit'
DRIVE_CHECKPOINTS = ORBIT_DRIVE / 'checkpoints'
DRIVE_DATA = ORBIT_DRIVE / 'data'
DRIVE_RESULTS = ORBIT_DRIVE / 'results'
DRIVE_LOGS = ORBIT_DRIVE / 'logs'

for p in [DRIVE_CHECKPOINTS, DRIVE_DATA, DRIVE_RESULTS, DRIVE_LOGS]:
    p.mkdir(parents=True, exist_ok=True)

print("[+] Google Drive configured successfully:")
print(f"    Master Drive Root:  {DRIVE_ROOT}")
print(f"    Orbit Checkpoints:  {DRIVE_CHECKPOINTS}")
print(f"    Orbit Data Shards:  {DRIVE_DATA}")

### Cell 3: Clean Repository Sync & Dependency Setup
Fetches the latest upstream commits cleanly (avoiding git merge/lock conflicts) and installs Crafter dependencies.

In [ ]:
# Cell 3: Repository Sync
%cd /content
import os
import sys

REPO_DIR = Path('/content/learn-a-world')
if (REPO_DIR / '.git').exists():
    %cd /content/learn-a-world
    !git fetch origin main
    !git reset --hard origin/main
else:
    !rm -rf /content/learn-a-world
    !git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world
    %cd /content/learn-a-world

# Set python search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.environ['PYTHONPATH'] = str(REPO_DIR)

# Install dependencies quietly
!pip install -q --upgrade pip
!pip install -q crafter gymnasium pytest numpy torchvision timm matplotlib
print("[+] Repository synced to latest commit:")
!git log -1 --oneline

### Cell 4: Checkpoint Discovery & Verification
Verifies that the Gate 6 H=4 policy checkpoint is available (searches repository and Google Drive). If missing, provides an interactive one-click upload.

In [ ]:
# Cell 4: Checkpoint Verification
from pathlib import Path
import shutil

candidate_checkpoints = [
    Path('results/gate6/policy_h4.pt'),
    Path('results/gate6/policy_h4.pt.gz'),
    DRIVE_CHECKPOINTS / 'policy_h4.pt',
    DRIVE_CHECKPOINTS / 'policy_h4.pt.gz',
    DRIVE_ROOT / 'results/gate6/policy_h4.pt',
    DRIVE_ROOT / 'results/gate6/policy_h4.pt.gz',
]

active_checkpoint = None
for c in candidate_checkpoints:
    if c.exists() and c.is_file():
        active_checkpoint = c
        break

if active_checkpoint:
    print(f"[+] Gate 6 H=4 policy candidate verified at: {active_checkpoint}")
    print(f"    File Size: {active_checkpoint.stat().st_size / (1024*1024):.2f} MB")
    # Ensure a local copy exists in results/gate6/ for standard relative path loading
    Path('results/gate6').mkdir(parents=True, exist_ok=True)
    if not Path('results/gate6/policy_h4.pt').exists() and not Path('results/gate6/policy_h4.pt.gz').exists():
        shutil.copy2(str(active_checkpoint), f"results/gate6/{active_checkpoint.name}")
else:
    print("[-] policy_h4 checkpoint not found in repo or Google Drive.")
    print("    Please upload policy_h4.pt or policy_h4.pt.gz using the widget below:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        dest = DRIVE_CHECKPOINTS / fname
        shutil.move(fname, str(dest))
        active_checkpoint = dest
        Path('results/gate6').mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(dest), f"results/gate6/{fname}")
        print(f"[+] Successfully saved uploaded checkpoint to: {dest}")

### Cell 5: Execute Phase 3A — Collect 30 Real Episodes & Validate Disk Reload
Runs the pure visual rollout collection, asserts transition schemas, validates boundary masks, and automatically writes the replay shard to both local disk and Google Drive.

In [ ]:
# Cell 5: Run Phase 3A
%cd /content/learn-a-world

!python scripts/run_phase3a_collection.py \
  --policy-checkpoint results/gate6/policy_h4.pt \
  --episodes 30 \
  --start-seed 1000 \
  --max-steps 300 \
  --temperature 1.2 \
  --device auto \
  --output data/phase3a_selfplay_30eps.npz \
  --drive-dir "{DRIVE_ROOT}"

### Cell 6: Visual Acceptance Report & Metrics Summary
Loads the verified report JSON and displays the acceptance checklist.

In [ ]:
# Cell 6: Display Phase 3A Acceptance Checklist
import json

report_path = Path('data/phase3a_verification_report.json')
if not report_path.exists():
    # Check Google Drive fallback
    report_path = DRIVE_DATA / 'phase3a_verification_report.json'

with open(report_path) as f:
    rep = json.load(f)

print("=" * 80)
print(f"PHASE 3A ACCEPTANCE REPORT: {rep['status']}")
print("=" * 80)
print(f"Episodes Collected:        {rep['num_episodes']}")
print(f"Total Frames:              {rep['total_frames']}")
print(f"Mean Return / Episode:     {rep['mean_return']:+.2f} ± {rep['std_return']:.2f}")
print(f"Mean Steps / Episode:      {rep['mean_steps']:.1f}")
print(f"Movement Percentage:       {rep['movement_percentage']:.1f}%")
print(f"Action Shannon Entropy:    {rep['action_entropy']:.3f}")
print(f"Replay File Size:          {rep['file_size_mb']:.2f} MB")
print(f"Local Replay Path:         {rep['output_file']}")
print(f"Drive Backup Path:         {DRIVE_DATA / 'phase3a_selfplay_30eps.npz'}")
print("-" * 80)
print("Acceptance Criteria Checklist:")
for item, val in rep['checklist'].items():
    print(f"  [{'PASS' if val else 'FAIL'}] {item.replace('_', ' ').capitalize()}")
print("=" * 80)